# Lesson 1.4 — CloudFormation Basics: Infrastructure as Code

**Level: Beginner**

## Learning Objectives

By the end of this lesson, you will be able to:

- Explain the structure of a CloudFormation template (YAML and JSON)
- Write a CloudFormation template that creates an S3 bucket and IAM role
- Deploy, update, and delete a CloudFormation stack using the CLI
- Understand stack drift detection and change sets

**Exam relevance:** SOA-C03 Domain 3 (Deployment, Provisioning, and Automation) — CloudFormation is heavily tested.

_Reference: [CloudFormation User Guide](https://docs.aws.amazon.com/cloudformation/latest/userguide/)_

## 1. What is AWS CloudFormation?

**CloudFormation** is AWS's native Infrastructure as Code (IaC) service. It lets you define AWS resources in a **template** (YAML or JSON), and CloudFormation manages the **deployment, update, and deletion** of those resources.

**Why IaC?**
- Reproducible: deploy the same infrastructure to dev/staging/prod consistently
- Version-controlled: templates live in Git alongside application code
- Auditable: every change is tracked
- Rollback: if a deployment fails, CloudFormation rolls back automatically

### Template Structure

```yaml
AWSTemplateFormatVersion: '2010-09-09'   # Always this value
Description: 'What this template does'  # Optional

Parameters:           # User inputs at deploy time
  EnvironmentName:
    Type: String
    Default: dev
    AllowedValues: [dev, staging, prod]

Mappings:             # Static lookup tables
  InstanceTypeMap:
    dev:  { InstanceType: t3.micro }
    prod: { InstanceType: m5.large }

Conditions:           # Conditional resource creation
  IsProd: !Equals [!Ref EnvironmentName, prod]

Resources:            # REQUIRED — the actual AWS resources
  MyBucket:
    Type: AWS::S3::Bucket
    Properties:
      BucketName: !Sub '${EnvironmentName}-my-app-bucket'

Outputs:              # Values to export or display after deployment
  BucketArn:
    Value: !GetAtt MyBucket.Arn
    Export:
      Name: !Sub '${AWS::StackName}-BucketArn'
```

### Key Intrinsic Functions

| Function | What it does | Example |
|----------|-------------|--------|
| `!Ref` | Reference a parameter or resource logical ID | `!Ref EnvironmentName` |
| `!Sub` | String substitution | `!Sub '${AWS::AccountId}-bucket'` |
| `!GetAtt` | Get attribute of a resource | `!GetAtt MyBucket.Arn` |
| `!If` | Conditional value | `!If [IsProd, 'm5.large', 't3.micro']` |
| `!Select` | Pick from list | `!Select [0, !GetAZs '']` |
| `!Join` | Join strings | `!Join [':', ['arn', 'aws', 's3']]` |
| `!ImportValue` | Import from another stack's export | `!ImportValue SharedVpcId` |

In [ ]:
import boto3
import subprocess
import json
import time

# Get account ID and region for use in templates
sts = boto3.client('sts')
identity = sts.get_caller_identity()
ACCOUNT_ID = identity['Account']
REGION = boto3.session.Session().region_name or 'us-east-1'

print(f"Account ID: {ACCOUNT_ID}")
print(f"Region: {REGION}")

In [ ]:
%%writefile cloudops-demo-stack.yaml
AWSTemplateFormatVersion: '2010-09-09'
Description: 'CloudOps Learning Demo Stack - S3 Bucket + IAM Role'

Parameters:
  EnvironmentName:
    Type: String
    Default: dev
    AllowedValues: [dev, staging, prod]
    Description: 'Deployment environment'

  ProjectName:
    Type: String
    Default: cloudops-learning
    Description: 'Project name used for resource naming'

Conditions:
  # Only enable versioning in prod
  IsProd: !Equals [!Ref EnvironmentName, prod]

Resources:
  # S3 Bucket for application artifacts
  ArtifactBucket:
    Type: AWS::S3::Bucket
    DeletionPolicy: Delete
    Properties:
      BucketName: !Sub '${ProjectName}-${EnvironmentName}-artifacts-${AWS::AccountId}'
      VersioningConfiguration:
        # Only enable versioning in production
        Status: !If [IsProd, Enabled, Suspended]
      BucketEncryption:
        ServerSideEncryptionConfiguration:
          - ServerSideEncryptionByDefault:
              SSEAlgorithm: AES256
      PublicAccessBlockConfiguration:
        BlockPublicAcls: true
        BlockPublicPolicy: true
        IgnorePublicAcls: true
        RestrictPublicBuckets: true
      Tags:
        - Key: Environment
          Value: !Ref EnvironmentName
        - Key: Project
          Value: !Ref ProjectName
        - Key: ManagedBy
          Value: CloudFormation

  # IAM Role for EC2 to access the S3 bucket
  EC2InstanceRole:
    Type: AWS::IAM::Role
    Properties:
      RoleName: !Sub '${ProjectName}-${EnvironmentName}-ec2-role'
      AssumeRolePolicyDocument:
        Version: '2012-10-17'
        Statement:
          - Effect: Allow
            Principal:
              Service: ec2.amazonaws.com
            Action: sts:AssumeRole
      Policies:
        - PolicyName: S3ArtifactAccess
          PolicyDocument:
            Version: '2012-10-17'
            Statement:
              - Effect: Allow
                Action:
                  - s3:GetObject
                  - s3:PutObject
                  - s3:ListBucket
                Resource:
                  - !GetAtt ArtifactBucket.Arn
                  - !Sub '${ArtifactBucket.Arn}/*'
      ManagedPolicyArns:
        - arn:aws:iam::aws:policy/AmazonSSMManagedInstanceCore
      Tags:
        - Key: Environment
          Value: !Ref EnvironmentName

  # Instance Profile wraps the role for EC2
  EC2InstanceProfile:
    Type: AWS::IAM::InstanceProfile
    Properties:
      InstanceProfileName: !Sub '${ProjectName}-${EnvironmentName}-ec2-profile'
      Roles:
        - !Ref EC2InstanceRole

Outputs:
  BucketName:
    Description: 'Name of the S3 artifact bucket'
    Value: !Ref ArtifactBucket
    Export:
      Name: !Sub '${AWS::StackName}-BucketName'

  BucketArn:
    Description: 'ARN of the S3 artifact bucket'
    Value: !GetAtt ArtifactBucket.Arn
    Export:
      Name: !Sub '${AWS::StackName}-BucketArn'

  InstanceProfileArn:
    Description: 'ARN of the EC2 Instance Profile'
    Value: !GetAtt EC2InstanceProfile.Arn
    Export:
      Name: !Sub '${AWS::StackName}-InstanceProfileArn'

In [ ]:
# Validate the template before deploying
# This catches syntax errors and invalid resource configurations

result = subprocess.run(
    'aws cloudformation validate-template --template-body file://cloudops-demo-stack.yaml',
    shell=True, capture_output=True, text=True
)

if result.returncode == 0:
    print("Template is VALID")
    response = json.loads(result.stdout)
    print("\nParameters declared:")
    for p in response.get('Parameters', []):
        print(f"  {p['ParameterKey']:20} (Default: {p.get('DefaultValue', 'none')})")
else:
    print(f"Template is INVALID:\n{result.stderr}")

In [ ]:
# Deploy the stack using a Change Set (best practice — preview before applying)
import boto3
import time

cf = boto3.client('cloudformation', region_name=REGION)

STACK_NAME = 'cloudops-learning-demo'
CHANGESET_NAME = 'initial-deployment'

# Read the template
with open('cloudops-demo-stack.yaml', 'r') as f:
    template_body = f.read()

# Create a Change Set (preview what will be created/modified/deleted)
try:
    # Determine if stack exists already
    try:
        cf.describe_stacks(StackName=STACK_NAME)
        changeset_type = 'UPDATE'
    except cf.exceptions.ClientError:
        changeset_type = 'CREATE'

    response = cf.create_change_set(
        StackName=STACK_NAME,
        TemplateBody=template_body,
        Parameters=[
            {'ParameterKey': 'EnvironmentName', 'ParameterValue': 'dev'},
            {'ParameterKey': 'ProjectName', 'ParameterValue': 'cloudops-learning'}
        ],
        Capabilities=['CAPABILITY_IAM', 'CAPABILITY_NAMED_IAM'],  # Required for IAM resources
        ChangeSetName=CHANGESET_NAME,
        ChangeSetType=changeset_type
    )
    print(f"Creating change set ({changeset_type})... waiting for review...")
    
    # Wait for change set to be ready
    waiter = cf.get_waiter('change_set_create_complete')
    waiter.wait(StackName=STACK_NAME, ChangeSetName=CHANGESET_NAME)
    
    # Describe the change set to see what will happen
    cs = cf.describe_change_set(StackName=STACK_NAME, ChangeSetName=CHANGESET_NAME)
    print(f"\nChange Set Summary ({len(cs['Changes'])} changes):")
    for change in cs['Changes']:
        rc = change['ResourceChange']
        print(f"  {rc['Action']:10} {rc['ResourceType']:40} {rc['LogicalResourceId']}")

except Exception as e:
    print(f"Change set creation: {e}")

In [ ]:
# Execute the Change Set to deploy the stack
import boto3

cf = boto3.client('cloudformation', region_name=REGION)
STACK_NAME = 'cloudops-learning-demo'
CHANGESET_NAME = 'initial-deployment'

try:
    cf.execute_change_set(StackName=STACK_NAME, ChangeSetName=CHANGESET_NAME)
    print("Executing change set... waiting for completion...")
    
    # Wait for stack to reach a stable state
    waiter = cf.get_waiter('stack_create_complete')
    try:
        waiter.wait(StackName=STACK_NAME, WaiterConfig={'MaxAttempts': 30, 'Delay': 10})
        print("Stack deployment SUCCESSFUL")
    except Exception:
        # Try update waiter too
        waiter = cf.get_waiter('stack_update_complete')
        waiter.wait(StackName=STACK_NAME)
        print("Stack update SUCCESSFUL")
    
    # Show stack outputs
    stack = cf.describe_stacks(StackName=STACK_NAME)['Stacks'][0]
    print("\nStack Outputs:")
    for output in stack.get('Outputs', []):
        print(f"  {output['OutputKey']:25} = {output['OutputValue']}")
except Exception as e:
    print(f"Stack execution error: {e}")

In [ ]:
# Stack Drift Detection — detect manual changes to CloudFormation-managed resources
# Drift = when actual resource config differs from the template

cf = boto3.client('cloudformation', region_name=REGION)
STACK_NAME = 'cloudops-learning-demo'

try:
    # Initiate drift detection
    response = cf.detect_stack_drift(StackName=STACK_NAME)
    detection_id = response['StackDriftDetectionId']
    print(f"Drift detection started. ID: {detection_id}")
    print("Waiting for detection to complete (10-30 seconds)...")
    
    # Poll until complete
    for _ in range(12):
        time.sleep(5)
        status = cf.describe_stack_drift_detection_status(StackDriftDetectionId=detection_id)
        if status['DetectionStatus'] not in ['DETECTION_IN_PROGRESS']:
            break
    
    drift_status = status.get('StackDriftStatus', 'UNKNOWN')
    print(f"\nDrift Status: {drift_status}")
    
    if drift_status == 'DRIFTED':
        drifted = cf.describe_stack_resource_drifts(
            StackName=STACK_NAME,
            StackResourceDriftStatusFilters=['MODIFIED', 'DELETED']
        )
        for drift in drifted['StackResourceDrifts']:
            print(f"  DRIFTED: {drift['LogicalResourceId']} ({drift['StackResourceDriftStatus']})")
    else:
        print("No drift detected — resources match the template.")
except Exception as e:
    print(f"Drift detection: {e}")

## Exercise

In [ ]:
%%writefile exercise-template.yaml
# TODO: Complete this CloudFormation template
# Requirements:
# 1. Add a Parameter called 'BucketSuffix' (Type: String, Default: 'logs')
# 2. Create an S3 bucket for logs named: 'cloudops-{EnvironmentName}-{BucketSuffix}-{AWS::AccountId}'
# 3. Enable lifecycle policy: transition to GLACIER after 90 days, expire after 365 days
# 4. Add an Output for the bucket name

AWSTemplateFormatVersion: '2010-09-09'
Description: 'Exercise: Log Bucket with Lifecycle'

Parameters:
  EnvironmentName:
    Type: String
    Default: dev
  # TODO: Add BucketSuffix parameter

Resources:
  LogBucket:
    Type: AWS::S3::Bucket
    Properties:
      # TODO: Add BucketName using !Sub
      # TODO: Add LifecycleConfiguration
      PublicAccessBlockConfiguration:
        BlockPublicAcls: true
        BlockPublicPolicy: true
        IgnorePublicAcls: true
        RestrictPublicBuckets: true

Outputs:
  # TODO: Add LogBucketName output

In [ ]:
# Cleanup — delete the demo stack
import boto3

cf = boto3.client('cloudformation', region_name=REGION)
STACK_NAME = 'cloudops-learning-demo'

try:
    cf.delete_stack(StackName=STACK_NAME)
    print(f"Deleting stack: {STACK_NAME}...")
    waiter = cf.get_waiter('stack_delete_complete')
    waiter.wait(StackName=STACK_NAME)
    print("Stack deleted successfully.")
except Exception as e:
    print(f"Cleanup: {e}")

# Also remove local template files
import os
for f in ['cloudops-demo-stack.yaml', 'exercise-template.yaml']:
    if os.path.exists(f):
        os.remove(f)
        print(f"Removed: {f}")

## Summary

- CloudFormation templates have 7 sections: `AWSTemplateFormatVersion`, `Description`, `Parameters`, `Mappings`, `Conditions`, `Resources`, `Outputs`
- **`Resources` is the only required section**
- Always use **Change Sets** before applying updates in production
- `CAPABILITY_IAM` and `CAPABILITY_NAMED_IAM` are required when templates create IAM resources
- **Drift detection** identifies manual changes to CloudFormation-managed resources
- `DeletionPolicy: Retain` protects resources from deletion when a stack is deleted

## Key Exam Points (SOA-C03)
- `!Sub` for string interpolation; `!GetAtt` for resource attributes; `!ImportValue` for cross-stack references
- CloudFormation **rollback** happens automatically on failure (stack goes to `ROLLBACK_COMPLETE`)
- **Stack sets** allow deployment across multiple accounts and regions (covered in Phase 4)

---

**Next Phase:** [Phase 2 — Monitoring, Logging & Observability](../phase-2-monitoring/lesson-2.1-cloudwatch-metrics-alarms.ipynb)